# A.1 Datakontroll

Det här notebooken genomför datakontroll inför den deskriptiva analysen och modelleringen. Kontrollerna syftar till att:

- bekräfta att datatyper och kategorinivåer är korrekta
- identifiera saknade värden
- undersöka extrema värden i numeriska variabler
- kontrollera att `Duration` ligger i rimligt intervall och identifiera eventuella nollor som påverkar log-offset-beräkningen

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

data_dir = Path("../../data")

df_train = pd.read_csv(data_dir / "Entreprenadförsäkring training.csv")
df_test = pd.read_csv(data_dir / "Entreprenadförsäkring test.csv")

print(f"Träning: {df_train.shape[0]:,} rader, {df_train.shape[1]} kolumner")
print(f"Test:    {df_test.shape[0]:,} rader, {df_test.shape[1]} kolumner")

FileNotFoundError: [Errno 2] No such file or directory: '../../data/Entreprenadförsäkring training.csv'

## 1. Datatyper och kategorinivåer

Här kontrolleras att varje kolumn har förväntad datatyp och att de kategoriska variablerna har rätt antal nivåer. Vi kontrollerar också om testfilen innehåller kategorivärd som inte förekommer i träningsfilen, vilket i så fall kan skapa problem vid modellering.

In [ ]:
dtype_summary = pd.DataFrame(
    {
        "Datatyp (träning)": df_train.dtypes.astype(str),
        "Datatyp (test)": df_test.dtypes.astype(str),
        "Unika värden (träning)": df_train.nunique(),
        "Unika värden (test)": df_test.nunique(),
    }
)

dtype_summary

,Datatyp (träning),Datatyp (test),Unika värden (träning),Unika värden (test)
Omsattning,float64,float64,74841,49256
Verksamhet,object,object,7,7
GeografisktOmrade,object,object,4,4
Ar,float64,float64,4,1
Forsakringsbelopp,float64,float64,15059,10047
Sjalvrisk,float64,float64,6,6
AntalSkador,int64,int64,3,3
Duration,float64,float64,283143,80018


In [3]:
cat_cols = ["Verksamhet", "GeografisktOmrade", "Ar"]

for col in cat_cols:
    train_levels = set(df_train[col].dropna().unique())
    test_levels = set(df_test[col].dropna().unique())
    only_in_test = test_levels - train_levels
    only_in_train = train_levels - test_levels

    print(f"--- {col} ---")
    print(f"  Nivåer i träning ({len(train_levels)}): {sorted(train_levels)}")
    print(f"  Nivåer i test    ({len(test_levels)}): {sorted(test_levels)}")
    if only_in_test:
        print(f"  *** Bara i test (okända nivåer): {sorted(only_in_test)} ***")
    if only_in_train:
        print(f"  Bara i träning (ej representerade i test): {sorted(only_in_train)}")
    print()

--- Verksamhet ---
  Nivåer i träning (7): ['Byggföretag', 'Elektriker', 'Grävning & Schaktning', 'Målare', 'Takarbeten', 'VVS', 'Övriga specialistföretag']
  Nivåer i test    (7): ['Byggföretag', 'Elektriker', 'Grävning & Schaktning', 'Målare', 'Takarbeten', 'VVS', 'Övriga specialistföretag']

--- GeografisktOmrade ---
  Nivåer i träning (4): ['Landsbyggd', 'Mellanstorstad', 'Småstad', 'Storstad']
  Nivåer i test    (4): ['Landsbyggd', 'Mellanstorstad', 'Småstad', 'Storstad']

--- Ar ---
  Nivåer i träning (4): [2021.0, 2022.0, 2023.0, 2024.0]
  Nivåer i test    (1): [2025.0]
  *** Bara i test (okända nivåer): [2025.0] ***
  Bara i träning (ej representerade i test): [2021.0, 2022.0, 2023.0, 2024.0]



### Tolkning

Notera eventuella avvikelser ovan. Om testfilen innehåller okända kategorinivåer måste detta hanteras i modelleringen, exempelvis via en `other`-kategori eller genom att kontrollera att modellens predict-funktion kan hantera osedda värden.

## 2. Saknade värden

Vi räknar antalet saknade värden per kolumn i både tränings- och testfilen.

In [4]:
def missing_summary(df: pd.DataFrame, label: str) -> pd.DataFrame:
    n = len(df)
    missing_count = df.isnull().sum()
    missing_pct = (missing_count / n * 100).round(3)
    return pd.DataFrame(
        {
            f"Saknade ({label})": missing_count,
            f"Andel % ({label})": missing_pct,
        }
    )


miss_train = missing_summary(df_train, "träning")
miss_test = missing_summary(df_test, "test")

missing_df = pd.concat([miss_train, miss_test], axis=1)
missing_df

,Saknade (träning),Andel % (träning),Saknade (test),Andel % (test)
Omsattning,0,0.0,0,0.0
Verksamhet,0,0.0,0,0.0
GeografisktOmrade,0,0.0,0,0.0
Ar,0,0.0,0,0.0
Forsakringsbelopp,0,0.0,0,0.0
Sjalvrisk,0,0.0,0,0.0
AntalSkador,0,0.0,0,0.0
Duration,0,0.0,0,0.0


### Tolkning

Om alla värden är noll saknas inga värden i datan, vilket är det förväntade utfallet för en väl förberedd dataset. Om saknade värden förekommer behöver vi besluta om imputation eller bortfall av dessa rader inför modelleringen.

## 3. Extremvärden i numeriska variabler

De ekonomiska variablerna `Omsattning`, `Forsakringsbelopp` och `Sjalvrisk` kan innehålla observationer som är extremt stora. Dessa kan vara genuint stora kontrakt eller potentiella datainmatningsfel. Vi undersöker min, max och höga percentiler för att bedöma detta.

In [5]:
numeric_cols = ["Omsattning", "Forsakringsbelopp", "Sjalvrisk"]


def extreme_summary(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in cols:
        s = df[col]
        rows.append(
            {
                "Variabel": col,
                "Min": s.min(),
                "P1": s.quantile(0.01),
                "P25": s.quantile(0.25),
                "Median": s.median(),
                "P75": s.quantile(0.75),
                "P99": s.quantile(0.99),
                "P99.9": s.quantile(0.999),
                "Max": s.max(),
                "Negativa": int((s < 0).sum()),
                "Nollor": int((s == 0).sum()),
            }
        )
    return pd.DataFrame(rows).set_index("Variabel")


print("=== Träning ===")
display(extreme_summary(df_train, numeric_cols))

print("\n=== Test ===")
display(extreme_summary(df_test, numeric_cols))

=== Träning ===


,Min,P1,P25,Median,P75,P99,P99.9,Max,Negativa,Nollor
Variabel,,,,,,,,,,
Omsattning,69000.0,790000.0,4127000.0,8098000.0,15904000.0,83468300.0,1.791769e+08,1.218919e+09,0,0
Forsakringsbelopp,50000.0,50000.0,223000.0,507000.0,1153000.0,8547000.0,2.128623e+07,1.199120e+08,0,0
Sjalvrisk,5000.0,5000.0,5000.0,10000.0,20000.0,50000.0,2.500000e+05,2.500000e+05,0,0



=== Test ===


,Min,P1,P25,Median,P75,P99,P99.9,Max,Negativa,Nollor
Variabel,,,,,,,,,,
Omsattning,105000.0,789000.0,4134000.0,8149000.0,15975000.0,82362520.0,1.798261e+08,870477000.0,0,0
Forsakringsbelopp,50000.0,50000.0,223000.0,507000.0,1156000.0,8389520.0,2.045175e+07,98414000.0,0,0
Sjalvrisk,5000.0,5000.0,5000.0,10000.0,20000.0,50000.0,2.500000e+05,250000.0,0,0


In [ ]:
for col in numeric_cols:
    threshold = df_train[col].quantile(0.999)
    top_rows = df_train[df_train[col] > threshold][[col, "Verksamhet", "GeografisktOmrade", "Ar", "AntalSkador"]]
    print(f"--- {col} > {threshold:,.0f} (topp 0,1 %, {len(top_rows)} rader) ---")
    print(top_rows.sort_values(col, ascending=False).head(10).to_string(index=False))
    print()

### Tolkning

De ekonomiska variablerna är tydligt högerskevda. Omsättningar och försäkringsbelopp i de högsta percentilerna representerar sannolikt genuint stora entreprenadobjekt snarare än datainmatningsfel, eftersom de följer ett rimligt mönster (se verksamhetstyp och år ovan).

**Åtgärd:** Inga rader tas bort på grund av extremvärden i dessa variabler. Vid modellering används log-transformering för att minska inverkan av extremt stora värden och för att linjäritetskravet i GLM bättre ska uppfyllas.

## 4. Duration: intervallkontroll och nollor

`Duration` mäter exponeringstiden i år och används som log-offset i Poisson-GLM: `offset = log(Duration)`. Det innebär att värden ≤ 0 gör log-beräkningen odefinierad och måste identifieras och hanteras explicit.

In [6]:
def duration_summary(df: pd.DataFrame, label: str) -> pd.DataFrame:
    d = df["Duration"]
    n = len(d)
    return pd.DataFrame(
        {
            "Mått": [
                "Min",
                "Max",
                "Medel",
                "Median",
                "Andel = 0 (%)",
                "Andel < 0 (%)",
                "Andel ≤ 0 (%)",
                "Andel = 1 (%)",
                "Andel > 1 (%)",
                "Antal exakt noll",
                "Antal negativa",
            ],
            label: [
                round(d.min(), 6),
                round(d.max(), 6),
                round(d.mean(), 4),
                round(d.median(), 4),
                round((d == 0).mean() * 100, 4),
                round((d < 0).mean() * 100, 4),
                round((d <= 0).mean() * 100, 4),
                round((d == 1).mean() * 100, 2),
                round((d > 1).mean() * 100, 4),
                int((d == 0).sum()),
                int((d < 0).sum()),
            ],
        }
    ).set_index("Mått")


dur_train = duration_summary(df_train, "Träning")
dur_test = duration_summary(df_test, "Test")

pd.concat([dur_train, dur_test], axis=1)

,Träning,Test
Mått,,
Min,0.095315,0.09532
Max,1.000000,1.00000
Medel,0.894200,0.89460
Median,1.000000,1.00000
Andel = 0 (%),0.000000,0.00000
Andel < 0 (%),0.000000,0.00000
Andel ≤ 0 (%),0.000000,0.00000
Andel = 1 (%),72.600000,72.56000
Andel > 1 (%),0.000000,0.00000


In [7]:
zero_dur_train = df_train[df_train["Duration"] <= 0]
zero_dur_test = df_test[df_test["Duration"] <= 0]

if len(zero_dur_train) > 0:
    print(f"Träning — {len(zero_dur_train)} rad(er) med Duration ≤ 0:")
    display(zero_dur_train.head(20))
else:
    print("Träning: inga rader med Duration ≤ 0.")

if len(zero_dur_test) > 0:
    print(f"\nTest — {len(zero_dur_test)} rad(er) med Duration ≤ 0:")
    display(zero_dur_test.head(20))
else:
    print("Test: inga rader med Duration ≤ 0.")

Träning: inga rader med Duration ≤ 0.
Test: inga rader med Duration ≤ 0.


In [8]:
gt1_train = df_train[df_train["Duration"] > 1]
gt1_test = df_test[df_test["Duration"] > 1]

print(f"Träning: {len(gt1_train):,} rader med Duration > 1 ({len(gt1_train)/len(df_train)*100:.3f} %)")
if len(gt1_train) > 0:
    print("  Min Duration > 1:", gt1_train["Duration"].min())
    print("  Max Duration > 1:", gt1_train["Duration"].max())

print(f"\nTest:    {len(gt1_test):,} rader med Duration > 1 ({len(gt1_test)/len(df_test)*100:.3f} %)")
if len(gt1_test) > 0:
    print("  Min Duration > 1:", gt1_test["Duration"].min())
    print("  Max Duration > 1:", gt1_test["Duration"].max())

Träning: 0 rader med Duration > 1 (0.000 %)

Test:    0 rader med Duration > 1 (0.000 %)


### Tolkning

**Förväntat intervall:** `Duration` bör ligga i intervallet (0, 1] för de flesta rader, eftersom det mäter andelen av ett kalenderår som en försäkring var aktiv. Värden exakt lika med 1 innebär en hel exponeringsperiod.

**Nollor och negativa värden:** Om sådana finns måste de hanteras innan log-offset beräknas. Rekommendation:
- Rader med `Duration <= 0` bör tas bort (troligtvis dataposter utan reell exponering) eller `Duration` sätts till ett litet golvvärde (t.ex. `1e-6`) om skadorna ska behållas.
- Antalet sådana rader dokumenteras i rapporten.

**Värden > 1:** Om sådana förekommer behöver vi bedöma om det är ett datafel (t.ex. att Duration anges i månader snarare än år för vissa rader) eller om det reflekterar fleråriga kontrakt. Antalet och storleken noteras ovan.

## Sammanfattning av datakontroll

| Kontroll | Resultat | Åtgärd |
|---|---|---|
| Datatyper | Se tabell ovan | Inga typer behöver konverteras manuellt |
| Kategorinivåer | Se utskrift ovan | Kontrollera okända nivåer i test vid modellering |
| Saknade värden | Se tabell ovan | Imputation eller bortfall vid behov |
| Extremvärden | Höger skevar fördelningar, inga troliga fel | Log-transformering i modellering |
| Duration ≤ 0 | Se tabell ovan | Bort- eller golvvärdeshantering |
| Duration > 1 | Se tabell ovan | Bedöm om datfel eller fleråriga kontrakt |